# Reuters 46-class classification: regularization experiments

Exploratory notebook (code unchanged from the original run; figures were removed from the saved outputs to keep the file small and are available in `results/`).

**Flow:** load data → multi-hot vectorization → baseline (64-64) → larger models that overfit (512-512, 128-128) → L2, L1, L1+L2, Dropout and Dropout+L2 experiments → grid comparison of 27 configurations (128-128, 30 epochs).

**Evaluation:** all numbers in this notebook are *validation* accuracies on a 1,000-sample hold-out set (`x_train[:1000]`); the test set is not used here (see notebook 02). The last cells re-run the same grid, so the numbers differ slightly between those cells because of random initialization.


In [1]:
from keras.datasets import reuters
import numpy as np

# بارگیری دیتاست با ۱۰٬۰۰۰ کلمه پرتکرار
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

# نمایش اطلاعات اولیه
print(f"تعداد داده‌های آموزش: {len(train_data)}")
print(f"تعداد داده‌های تست: {len(test_data)}")
print(f"تعداد دسته‌ها (کلاس‌ها): {np.max(train_labels) + 1}")




تعداد داده‌های آموزش: 8982
تعداد داده‌های تست: 2246
تعداد دسته‌ها (کلاس‌ها): 46


فرضیات اولیه و بردارسازی داده‌ها


In [2]:
# بردارسازی داده‌های ورودی (one-hot)
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# برچسب‌ها به صورت one-hot
from keras.utils import to_categorical

one_hot_train_labels = to_categorical(train_labels)
one_hot_test_labels = to_categorical(test_labels)


 انتخاب معیار موفقیت و تابع زیان

In [3]:
from keras import models, layers

# مدل اولیه ساده‌تر از مدل نهایی
model = models.Sequential()
model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(46, activation='softmax'))  # 46 دسته خبری

# کامپایل مدل با categorical crossentropy
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


تصمیم‌گیری درباره پروتکل ارزیابی

In [4]:
# تقسیم بخشی از داده‌های آموزش برای اعتبارسنجی
x_val = x_train[:1000]
partial_x_train = x_train[1000:]

y_val = one_hot_train_labels[:1000]
partial_y_train = one_hot_train_labels[1000:]


آماده‌سازی داده‌ها و آموزش مدل پایه

In [5]:
# آموزش مدل پایه با ۲۰ دوره
history = model.fit(partial_x_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(x_val, y_val))


Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.4110 - loss: 3.1741 - val_accuracy: 0.6200 - val_loss: 1.7691
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6637 - loss: 1.5976 - val_accuracy: 0.6940 - val_loss: 1.3609
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7367 - loss: 1.2038 - val_accuracy: 0.7310 - val_loss: 1.1869
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7873 - loss: 0.9866 - val_accuracy: 0.7480 - val_loss: 1.0802
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8219 - loss: 0.8131 - val_accuracy: 0.7700 - val_loss: 1.0070
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8562 - loss: 0.6515 - val_accuracy: 0.7860 - val_loss: 0.9604
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8802 - loss: 0.5503 - val_accuracy: 0.7970 - val_loss: 0.9384
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9072 - loss: 0.4412 - val_accuracy: 0.8050 - v

نمایش نمودار دقت و زیان:

In [6]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

# نمودار دقت
plt.figure(figsize=(10, 4))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, color='navy', marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, color='pink', marker='o', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, color='navy', marker='o', label='Training Loss')
plt.plot(epochs, val_loss, color='pink', marker='o', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()



In [9]:
# مدل با ظرفیت بیشتر برای نمایش overfitting
overfit_model = models.Sequential()
overfit_model.add(layers.Dense(512, activation='relu', input_shape=(10000,)))
overfit_model.add(layers.Dense(512, activation='relu'))
overfit_model.add(layers.Dense(46, activation='softmax'))

overfit_model.compile(optimizer='rmsprop',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

overfit_history = overfit_model.fit(partial_x_train, partial_y_train,
                                    epochs=40, batch_size=512,
                                    validation_data=(x_val, y_val))

import matplotlib.pyplot as plt

acc = overfit_history.history['accuracy']
val_acc = overfit_history.history['val_accuracy']
loss = overfit_history.history['loss']
val_loss = overfit_history.history['val_loss']

epochs = range(1, len(acc) + 1)
print(epochs)
# نمودار دقت
plt.figure(figsize=(10, 4))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, color='deepskyblue', marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, color='deeppink', marker='o', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, color='deepskyblue', marker='o', label='Training Loss')
plt.plot(epochs, val_loss, color='deeppink', marker='o', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

Epoch 1/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.4091 - loss: 2.5692 - val_accuracy: 0.6610 - val_loss: 1.4114
Epoch 2/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.7247 - loss: 1.1854 - val_accuracy: 0.7580 - val_loss: 1.0572
Epoch 3/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.8093 - loss: 0.8256 - val_accuracy: 0.7760 - val_loss: 0.9700
Epoch 4/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.8743 - loss: 0.5648 - val_accuracy: 0.7900 - val_loss: 0.8920
Epoch 5/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.9058 - loss: 0.3996 - val_accuracy: 0.8110 - val_loss: 0.8790
Epoch 6/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.9327 - loss: 0.2985 - val_accuracy: 0.8230 - val_loss: 0.7826
Epoch 7/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.9438 - loss: 0.2216 - val_accuracy: 0.8060 - val_loss: 0.8976
Epoch 8/40
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.9510 - loss: 0.1942 - val_accuracy: 0.8080 - v

 افزایش پیچیدگی و مشاهده overfitting

In [10]:
# مدل با ظرفیت بیشتر برای نمایش overfitting
overfit_model = models.Sequential()
overfit_model.add(layers.Dense(128, activation='relu', input_shape=(10000,)))
overfit_model.add(layers.Dense(128, activation='relu'))
overfit_model.add(layers.Dense(46, activation='softmax'))

overfit_model.compile(optimizer='rmsprop',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

overfit_history = overfit_model.fit(partial_x_train, partial_y_train,
                                    epochs=30, batch_size=512,
                                    validation_data=(x_val, y_val))

import matplotlib.pyplot as plt

acc = overfit_history.history['accuracy']
val_acc = overfit_history.history['val_accuracy']
loss = overfit_history.history['loss']
val_loss = overfit_history.history['val_loss']

epochs = range(1, len(acc) + 1)

# نمودار دقت
plt.figure(figsize=(10, 4))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, color='deepskyblue', marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, color='deeppink', marker='o', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, color='deepskyblue', marker='o', label='Training Loss')
plt.plot(epochs, val_loss, color='deeppink', marker='o', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()



Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.3739 - loss: 2.9431 - val_accuracy: 0.6630 - val_loss: 1.5405
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6913 - loss: 1.3967 - val_accuracy: 0.7130 - val_loss: 1.2364
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7753 - loss: 1.0180 - val_accuracy: 0.7700 - val_loss: 1.0796
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8349 - loss: 0.7669 - val_accuracy: 0.7930 - val_loss: 0.9672
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8708 - loss: 0.6079 - val_accuracy: 0.8020 - val_loss: 0.9054
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8931 - loss: 0.4871 - val_accuracy: 0.7920 - val_loss: 0.9078
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9250 - loss: 0.3651 - val_accuracy: 0.8090 - val_loss: 0.8622
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9336 - loss: 0.3087 - val_accuracy: 0.8140 - v

اعمال Regularization و تنظیم ابرپارامترها


L2 Regularization

In [16]:
import tensorflow as tf
from tensorflow.keras import models, layers, regularizers
import matplotlib.pyplot as plt
import numpy as np

# فرض می‌کنیم partial_x_train, partial_y_train, x_val, y_val از قبل تعریف شده‌اند

# لیست مقادیر مختلف L2
l2_values = [0.0001, 0.01, 0.001]

# ذخیره تاریخچه‌ها و بهترین دقت اعتبارسنجی برای هر مقدار L2
histories = {}
best_val_accuracies = {}

# حلقه برای آزمایش مقادیر مختلف L2
for l2_val in l2_values:
    print(f"\nTraining model with L2={l2_val}")
    
    # تعریف مدل
    l2_model = models.Sequential()
    l2_model.add(layers.Dense(64, activation='relu',
                             kernel_regularizer=regularizers.l2(l2_val),
                             input_shape=(10000,)))
    l2_model.add(layers.Dense(64, activation='relu',
                             kernel_regularizer=regularizers.l2(l2_val)))
    l2_model.add(layers.Dense(46, activation='softmax'))

    # کامپایل مدل
    l2_model.compile(optimizer='rmsprop',
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

    # آموزش مدل
    history = l2_model.fit(partial_x_train, partial_y_train,
                          epochs=20, batch_size=512,
                          validation_data=(x_val, y_val),
                          verbose=1)
    
    # ذخیره تاریخچه
    histories[l2_val] = history.history
    
    # ذخیره بهترین دقت اعتبارسنجی
    best_val_accuracies[l2_val] = max(history.history['val_accuracy'])

# پیدا کردن بهترین مقدار L2
best_l2 = max(best_val_accuracies, key=best_val_accuracies.get)
print(f"\nBest L2 value: {best_l2} with validation accuracy: {best_val_accuracies[best_l2]:.4f}")

# رسم نمودارهای جداگانه برای هر مقدار L2
for l2_val in l2_values:
    history = histories[l2_val]
    
    plt.figure(figsize=(12, 4))
    
    # نمودار دقت
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='Train Accuracy')
    plt.plot(history['val_accuracy'], '--', label='Validation Accuracy')
    plt.title(f'Accuracy (L2={l2_val})')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # نمودار خطا
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='Train Loss')
    plt.plot(history['val_loss'], '--', label='Validation Loss')
    plt.title(f'Loss (L2={l2_val})')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

# رسم نمودار مقایسه‌ای برای همه مقادیر L2
plt.figure(figsize=(12, 8))

# نمودار دقت اعتبارسنجی
plt.subplot(2, 1, 1)
for l2_val in l2_values:
    plt.plot(histories[l2_val]['val_accuracy'], label=f'Val Acc (L2={l2_val})')
plt.title('Validation Accuracy Comparison for Different L2 Values')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)

# نمودار خطای اعتبارسنجی
plt.subplot(2, 1, 2)
for l2_val in l2_values:
    plt.plot(histories[l2_val]['val_loss'], label=f'Val Loss (L2={l2_val})')
plt.title('Validation Loss Comparison for Different L2 Values')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


Training model with L2=0.0001
Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.3918 - loss: 3.2232 - val_accuracy: 0.6350 - val_loss: 1.8778
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6743 - loss: 1.6716 - val_accuracy: 0.6830 - val_loss: 1.4259
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7274 - loss: 1.2488 - val_accuracy: 0.7290 - val_loss: 1.2225
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7911 - loss: 0.9954 - val_accuracy: 0.7540 - val_loss: 1.1208
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8133 - loss: 0.8710 - val_accuracy: 0.7780 - val_loss: 1.0376
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8449 - loss: 0.7287 - val_accuracy: 0.7810 - val_loss: 1.0036
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8677 - loss: 0.6458 - val_accuracy: 0.8070 - val_loss: 0.9603
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8965 - loss: 0.

L2 Regularization + Dropout

In [17]:
import tensorflow as tf
from tensorflow.keras import models, layers, regularizers
import matplotlib.pyplot as plt
import numpy as np

# فرض می‌کنیم partial_x_train, partial_y_train, x_val, y_val از قبل تعریف شده‌اند

# لیست مقادیر مختلف L2 و Dropout
l2_values = [0.0001, 0.001, 0.01, 0.1]
dropout_rates = [0.3, 0.5, 0.7]

# ذخیره تاریخچه‌ها و بهترین دقت اعتبارسنجی برای هر ترکیب
histories = {}
best_val_accuracies = {}

# حلقه برای آزمایش ترکیب‌های مختلف L2 و Dropout
for l2_val in l2_values:
    for dropout_rate in dropout_rates:
        combo_key = f"L2={l2_val}, Dropout={dropout_rate}"
        print(f"\nTraining model with {combo_key}")
        
        # تعریف مدل
        combo_model = models.Sequential()
        combo_model.add(layers.Dense(64, activation='relu',
                                    kernel_regularizer=regularizers.l2(l2_val),
                                    input_shape=(10000,)))
        combo_model.add(layers.Dropout(dropout_rate))
        combo_model.add(layers.Dense(64, activation='relu',
                                    kernel_regularizer=regularizers.l2(l2_val)))
        combo_model.add(layers.Dropout(dropout_rate))
        combo_model.add(layers.Dense(46, activation='softmax'))

        # کامپایل مدل
        combo_model.compile(optimizer='rmsprop',
                           loss='categorical_crossentropy',
                           metrics=['accuracy'])

        # آموزش مدل
        history = combo_model.fit(partial_x_train, partial_y_train,
                                 epochs=20, batch_size=512,
                                 validation_data=(x_val, y_val),
                                 verbose=1)
        
        # ذخیره تاریخچه
        histories[combo_key] = history.history
        
        # ذخیره بهترین دقت اعتبارسنجی
        best_val_accuracies[combo_key] = max(history.history['val_accuracy'])

# پیدا کردن بهترین ترکیب
best_combo = max(best_val_accuracies, key=best_val_accuracies.get)
print(f"\nBest combination: {best_combo} with validation accuracy: {best_val_accuracies[best_combo]:.4f}")

# رسم نمودارهای جداگانه برای هر ترکیب
for combo_key in histories:
    history = histories[combo_key]
    
    plt.figure(figsize=(12, 4))
    
    # نمودار دقت
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='Train Accuracy')
    plt.plot(history['val_accuracy'], '--', label='Validation Accuracy')
    plt.title(f'Accuracy ({combo_key})')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # نمودار خطا
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='Train Loss')
    plt.plot(history['val_loss'], '--', label='Validation Loss')
    plt.title(f'Loss ({combo_key})')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

# رسم نمودار مقایسه‌ای برای همه ترکیب‌ها
plt.figure(figsize=(12, 8))

# نمودار دقت اعتبارسنجی
plt.subplot(2, 1, 1)
for combo_key in histories:
    plt.plot(histories[combo_key]['val_accuracy'], label=combo_key)
plt.title('Validation Accuracy Comparison for Different L2 and Dropout Combinations')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)

# نمودار خطای اعتبارسنجی
plt.subplot(2, 1, 2)
for combo_key in histories:
    plt.plot(histories[combo_key]['val_loss'], label=combo_key)
plt.title('Validation Loss Comparison for Different L2 and Dropout Combinations')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


Training model with L2=0.0001, Dropout=0.3
Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2903 - loss: 3.4024 - val_accuracy: 0.5960 - val_loss: 2.0169
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5816 - loss: 1.9816 - val_accuracy: 0.6560 - val_loss: 1.5279
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6596 - loss: 1.5826 - val_accuracy: 0.6950 - val_loss: 1.3240
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6968 - loss: 1.3295 - val_accuracy: 0.7190 - val_loss: 1.2228
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7192 - loss: 1.2223 - val_accuracy: 0.7330 - val_loss: 1.1567
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7449 - loss: 1.1075 - val_accuracy: 0.7570 - val_loss: 1.0990
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7689 - loss: 1.0090 - val_accuracy: 0.7610 - val_loss: 1.0625
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.79

L1-L2 Regularization

In [20]:
import tensorflow as tf
from tensorflow.keras import models, layers, regularizers
import matplotlib.pyplot as plt
import numpy as np

# فرض می‌کنیم partial_x_train, partial_y_train, x_val, y_val از قبل تعریف شده‌اند

# لیست مقادیر مختلف L1 و L2
l1_values = [0.0001, 0.001, 0.01]
l2_values = [0.0001, 0.001, 0.01]

# ذخیره تاریخچه‌ها و بهترین دقت اعتبارسنجی برای هر ترکیب
histories = {}
best_val_accuracies = {}

# حلقه برای آزمایش ترکیب‌های مختلف L1 و L2
for l1_val in l1_values:
    for l2_val in l2_values:
        combo_key = f"L1={l1_val}, L2={l2_val}"
        print(f"\nTraining model with {combo_key}")
        
        # تعریف مدل
        elastic_model = models.Sequential()
        elastic_model.add(layers.Dense(64, activation='relu',
                                      kernel_regularizer=regularizers.l1_l2(l1=l1_val, l2=l2_val),
                                      input_shape=(10000,)))
        elastic_model.add(layers.Dense(64, activation='relu',
                                      kernel_regularizer=regularizers.l1_l2(l1=l1_val, l2=l2_val)))
        elastic_model.add(layers.Dense(46, activation='softmax'))

        # کامپایل مدل
        elastic_model.compile(optimizer='rmsprop',
                             loss='categorical_crossentropy',
                             metrics=['accuracy'])

        # آموزش مدل
        history = elastic_model.fit(partial_x_train, partial_y_train,
                                   epochs=20, batch_size=512,
                                   validation_data=(x_val, y_val),
                                   verbose=1)
        
        # ذخیره تاریخچه
        histories[combo_key] = history.history
        
        # ذخیره بهترین دقت اعتبارسنجی
        best_val_accuracies[combo_key] = max(history.history['val_accuracy'])

# پیدا کردن بهترین ترکیب
best_combo = max(best_val_accuracies, key=best_val_accuracies.get)
print(f"\nBest combination: {best_combo} with validation accuracy: {best_val_accuracies[best_combo]:.4f}")

for combo_key in histories:
    history = histories[combo_key]
    
    # استخراج مقادیر
    train_acc = history['accuracy']
    val_acc = history['val_accuracy']
    train_loss = history['loss']
    val_loss = history['val_loss']
    
    # چاپ مقادیر برای هر epoch
    print(f"\nResults for {combo_key}:")
    print("Epoch | Train Acc | Val Acc | Train Loss | Val Loss")
    print("-" * 50)
    for epoch in range(len(train_acc)):
        print(f"{epoch+1:5d} | {train_acc[epoch]:.4f}   | {val_acc[epoch]:.4f} | {train_loss[epoch]:.4f}    | {val_loss[epoch]:.4f}")
        
# رسم نمودارهای جداگانه برای هر ترکیب
for combo_key in histories:
    history = histories[combo_key]
    
    plt.figure(figsize=(12, 4))
    
    # نمودار دقت
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='Train Accuracy')
    plt.plot(history['val_accuracy'], '--', label='Validation Accuracy')
    plt.title(f'Accuracy ({combo_key})')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # نمودار خطا
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='Train Loss')
    plt.plot(history['val_loss'], '--', label='Validation Loss')
    plt.title(f'Loss ({combo_key})')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

# رسم نمودار مقایسه‌ای برای همه ترکیب‌ها
plt.figure(figsize=(12, 8))

# نمودار دقت اعتبارسنجی
plt.subplot(2, 1, 1)
for combo_key in histories:
    plt.plot(histories[combo_key]['val_accuracy'], label=combo_key)
plt.title('Validation Accuracy Comparison for Different L1 and L2 Combinations')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)

# نمودار خطای اعتبارسنجی
plt.subplot(2, 1, 2)
for combo_key in histories:
    plt.plot(histories[combo_key]['val_loss'], label=combo_key)
plt.title('Validation Loss Comparison for Different L1 and L2 Combinations')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


Training model with L1=0.0001, L2=0.0001
Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.3331 - loss: 4.0175 - val_accuracy: 0.6360 - val_loss: 2.4865
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6637 - loss: 2.2589 - val_accuracy: 0.6790 - val_loss: 1.8980
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7248 - loss: 1.7128 - val_accuracy: 0.7330 - val_loss: 1.6041
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7702 - loss: 1.4267 - val_accuracy: 0.7550 - val_loss: 1.4478
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7938 - loss: 1.2773 - val_accuracy: 0.7700 - val_loss: 1.3450
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8138 - loss: 1.1379 - val_accuracy: 0.7890 - val_loss: 1.3059
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8353 - loss: 1.0551 - val_accuracy: 0.7860 - val_loss: 1.2879
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8454

L1 Regularization

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import models, layers, regularizers

# فرض: partial_x_train, partial_y_train, x_val, y_val قبلاً آماده شده‌اند

l1_values = [0.0001, 0.001, 0.01]
histories = {}

for l1_value in l1_values:
    print(f"\nTraining model with L1 = {l1_value}")
    model = models.Sequential()
    model.add(layers.Dense(64, activation='relu',
                           kernel_regularizer=regularizers.l1(l1_value),
                           input_shape=(10000,)))
    model.add(layers.Dense(64, activation='relu',
                           kernel_regularizer=regularizers.l1(l1_value)))
    model.add(layers.Dense(46, activation='softmax'))

    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(partial_x_train, partial_y_train,
                        epochs=20, batch_size=512,
                        validation_data=(x_val, y_val),
                        verbose=0)
    
    histories[l1_value] = history


# رسم نمودار جداگانه برای هر مقدار L1
epochs = range(1, 21)

for l1_value, history in histories.items():
    plt.figure(figsize=(14, 5))
    plt.suptitle(f"L1 Regularization = {l1_value}", fontsize=14)

    # دقت (Accuracy)
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history['accuracy'], 'bo--', label='Train Acc')
    plt.plot(epochs, history.history['val_accuracy'], 'r^-', label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # خطا (Loss)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history['loss'], 'bo--', label='Train Loss')
    plt.plot(epochs, history.history['val_loss'], 'r^-', label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # چاپ مقادیر فقط مربوط به آخرین ایپاک (Epoch 20)
    last_epoch = 19  # چون ایپاک‌ها از 0 شروع می‌شوند
    print(f"\nL1 = {l1_value}")
    print("Epoch\tTrain Acc\tVal Acc\t\tTrain Loss\tVal Loss")
    print(f"20\t{history.history['accuracy'][last_epoch]:.4f}\t\t"
          f"{history.history['val_accuracy'][last_epoch]:.4f}\t\t"
          f"{history.history['loss'][last_epoch]:.4f}\t\t{history.history['val_loss'][last_epoch]:.4f}")



Training model with L1 = 0.0001

Training model with L1 = 0.001

Training model with L1 = 0.01



L1 = 0.0001
Epoch	Train Acc	Val Acc		Train Loss	Val Loss
20	0.9312		0.8100		0.6179		1.1867



L1 = 0.001
Epoch	Train Acc	Val Acc		Train Loss	Val Loss
20	0.7667		0.7570		1.7959		1.8204



L1 = 0.01
Epoch	Train Acc	Val Acc		Train Loss	Val Loss
20	0.5588		0.5770		5.0314		4.9761


In [24]:
# رسم نمودارهای جداگانه برای هر ترکیب
for combo_key in histories:
    history = histories[combo_key]
    
    plt.figure(figsize=(12, 4))
    
    # نمودار دقت
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
    plt.plot(history.history['val_accuracy'], '--', label='Validation Accuracy', color='red')
    plt.title(f'Accuracy ({combo_key})')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # نمودار خطا
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', color='blue')
    plt.plot(history.history['val_loss'], '--', label='Validation Loss', color='red')
    plt.title(f'Loss ({combo_key})')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

# # رسم نمودار مقایسه‌ای برای همه ترکیب‌ها
# plt.figure(figsize=(12, 8))

# # نمودار دقت اعتبارسنجی
# plt.subplot(2, 1, 1)
# for combo_key in histories:
#     plt.plot(histories[combo_key].history['val_accuracy'], label=combo_key)
# plt.title('Validation Accuracy Comparison for Different L1 and L2 Combinations')
# plt.xlabel('Epoch')
# plt.ylabel('Validation Accuracy')
# plt.legend(loc='best')
# plt.grid(True)

# # نمودار خطای اعتبارسنجی
# plt.subplot(2, 1, 2)
# for combo_key in histories:
#     plt.plot(histories[combo_key].history['val_loss'], label=combo_key)
# plt.title('Validation Loss Comparison for Different L1 and L2 Combinations')
# plt.xlabel('Epoch')
# plt.ylabel('Validation Loss')
# plt.legend(loc='best')
# plt.grid(True)

# plt.tight_layout()
# plt.show()

Dropout

In [25]:
import matplotlib.pyplot as plt
from tensorflow.keras import models, layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import reuters
import numpy as np

# بارگذاری داده‌ها
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=10000)

# تبدیل به one-hot
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(x_train)
x_test = vectorize_sequences(x_test)

one_hot_train_labels = to_categorical(y_train)
one_hot_test_labels = to_categorical(y_test)

# تقسیم داده‌های آموزش به train و validation
x_val = x_train[:1000]
partial_x_train = x_train[1000:]

y_val = one_hot_train_labels[:1000]
partial_y_train = one_hot_train_labels[1000:]

# تابع برای آموزش و رسم نمودار
def train_and_plot(dropout_rates, neurons, epochs):
    history_dict = {}
    
    # ساخت مدل
    def build_model(dropout_rate):
        model = models.Sequential()
        model.add(layers.Dense(neurons, activation='relu', input_shape=(10000,)))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(neurons, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])
        return model
    
    # آموزش مدل‌ها
    for rate in dropout_rates:
        print(f"\nTraining with dropout rate = {rate}, neurons = {neurons}")
        model = build_model(rate)
        history = model.fit(
            partial_x_train, partial_y_train,
            epochs=epochs,
            batch_size=512,
            validation_data=(x_val, y_val),
            verbose=0
        )
        history_dict[rate] = history
    
    # رسم نمودارهای جداگانه برای هر نرخ Dropout
    for rate in dropout_rates:
        history = history_dict[rate].history
        
        # چاپ مقادیر برای هر epoch
        print(f"\nResults for Dropout Rate = {rate}:")
        print("Epoch | Train Acc | Val Acc | Train Loss | Val Loss")
        print("-" * 50)
        for epoch in range(len(history['accuracy'])):
            print(f"{epoch+1:5d} | {history['accuracy'][epoch]:.4f}   | {history['val_accuracy'][epoch]:.4f} | {history['loss'][epoch]:.4f}    | {history['val_loss'][epoch]:.4f}")
        
        # رسم نمودارهای جداگانه
        plt.figure(figsize=(12, 4))
        
        # نمودار دقت
        plt.subplot(1, 2, 1)
        plt.plot(history['accuracy'], label='Train Accuracy', color='blue')
        plt.plot(history['val_accuracy'], '--', label='Validation Accuracy', color='red')
        plt.title(f'Accuracy (Dropout={rate})')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)
        
        # نمودار خطا
        plt.subplot(1, 2, 2)
        plt.plot(history['loss'], label='Train Loss', color='blue')
        plt.plot(history['val_loss'], '--', label='Validation Loss', color='red')
        plt.title(f'Loss (Dropout={rate})')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        
        plt.tight_layout()
        plt.show()
    
    # رسم نمودار مقایسه‌ای برای همه نرخ‌های Dropout
    plt.figure(figsize=(12, 8))
    
    # نمودار دقت اعتبارسنجی
    plt.subplot(2, 1, 1)
    for rate in dropout_rates:
        plt.plot(history_dict[rate].history['val_accuracy'], label=f'Dropout={rate}')
    plt.title(f'Validation Accuracy Comparison (Neurons={neurons})')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Accuracy')
    plt.legend(loc='best')
    plt.grid(True)
    
    # نمودار خطای اعتبارسنجی
    plt.subplot(2, 1, 2)
    for rate in dropout_rates:
        plt.plot(history_dict[rate].history['val_loss'], label=f'Dropout={rate}')
    plt.title(f'Validation Loss Comparison (Neurons={neurons})')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Loss')
    plt.legend(loc='best')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

# اجرای آموزش برای مدل با 64 نورون (20 epoch)
print("Training Model with 64 Neurons")
train_and_plot(dropout_rates=[0.3, 0.5, 0.7], neurons=64, epochs=20)

# اجرای آموزش برای مدل با 128 نورون (30 epoch)
print("\nTraining Model with 128 Neurons")
train_and_plot(dropout_rates=[0.3, 0.5, 0.7], neurons=128, epochs=30)

Training Model with 64 Neurons

Training with dropout rate = 0.3, neurons = 64

Training with dropout rate = 0.5, neurons = 64

Training with dropout rate = 0.7, neurons = 64

Results for Dropout Rate = 0.3:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4519   | 0.6030 | 2.7184    | 1.7797
    2 | 0.6077   | 0.6590 | 1.7233    | 1.4351
    3 | 0.6734   | 0.7140 | 1.4196    | 1.2777
    4 | 0.7108   | 0.7290 | 1.2460    | 1.1741
    5 | 0.7387   | 0.7600 | 1.1068    | 1.1056
    6 | 0.7656   | 0.7680 | 1.0007    | 1.0427
    7 | 0.7885   | 0.7710 | 0.9021    | 1.0139
    8 | 0.8048   | 0.7800 | 0.8232    | 0.9803
    9 | 0.8182   | 0.7870 | 0.7568    | 0.9509
   10 | 0.8339   | 0.8030 | 0.6986    | 0.9280
   11 | 0.8480   | 0.7990 | 0.6355    | 0.9258
   12 | 0.8589   | 0.8070 | 0.5827    | 0.9162
   13 | 0.8633   | 0.8140 | 0.5502    | 0.8894
   14 | 0.8758   | 0.8080 | 0.5026    | 0.9088
   15 | 0.8846   | 0.8080 | 0.4


Results for Dropout Rate = 0.5:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.3321   | 0.5360 | 3.1615    | 2.1764
    2 | 0.5194   | 0.5950 | 2.1395    | 1.6571
    3 | 0.5784   | 0.6560 | 1.7738    | 1.4747
    4 | 0.6190   | 0.6800 | 1.5933    | 1.3716
    5 | 0.6518   | 0.6980 | 1.4552    | 1.2832
    6 | 0.6765   | 0.7120 | 1.3487    | 1.2337
    7 | 0.6949   | 0.7220 | 1.2574    | 1.1793
    8 | 0.7056   | 0.7310 | 1.1985    | 1.1449
    9 | 0.7229   | 0.7390 | 1.1443    | 1.1203
   10 | 0.7301   | 0.7480 | 1.0776    | 1.0904
   11 | 0.7434   | 0.7580 | 1.0225    | 1.0739
   12 | 0.7537   | 0.7680 | 0.9823    | 1.0543
   13 | 0.7641   | 0.7740 | 0.9331    | 1.0329
   14 | 0.7724   | 0.7830 | 0.8975    | 1.0118
   15 | 0.7830   | 0.7820 | 0.8549    | 1.0138
   16 | 0.7952   | 0.7890 | 0.8073    | 1.0096
   17 | 0.7965   | 0.7900 | 0.7956    | 0.9960
   18 | 0.8109   | 0.7890 | 0.7410    | 1.0107
   19 | 0.8140   |


Results for Dropout Rate = 0.7:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.2326   | 0.4390 | 3.3269    | 2.4975
    2 | 0.3899   | 0.5440 | 2.5646    | 1.9465
    3 | 0.4483   | 0.6020 | 2.2506    | 1.7673
    4 | 0.4975   | 0.6540 | 2.0186    | 1.6296
    5 | 0.5223   | 0.6580 | 1.8836    | 1.5495
    6 | 0.5526   | 0.6720 | 1.7927    | 1.4626
    7 | 0.5750   | 0.6880 | 1.7075    | 1.4160
    8 | 0.5965   | 0.6930 | 1.6258    | 1.3852
    9 | 0.6025   | 0.7030 | 1.5963    | 1.3483
   10 | 0.6195   | 0.7030 | 1.5255    | 1.3229
   11 | 0.6309   | 0.7050 | 1.4860    | 1.3049
   12 | 0.6392   | 0.7050 | 1.4406    | 1.2785
   13 | 0.6455   | 0.7070 | 1.4115    | 1.2689
   14 | 0.6587   | 0.7080 | 1.3750    | 1.2540
   15 | 0.6612   | 0.7110 | 1.3189    | 1.2383
   16 | 0.6604   | 0.7160 | 1.3111    | 1.2388
   17 | 0.6778   | 0.7200 | 1.2865    | 1.2324
   18 | 0.6773   | 0.7210 | 1.2786    | 1.2333
   19 | 0.6843   |


Training Model with 128 Neurons

Training with dropout rate = 0.3, neurons = 128

Training with dropout rate = 0.5, neurons = 128

Training with dropout rate = 0.7, neurons = 128

Results for Dropout Rate = 0.3:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4544   | 0.6360 | 2.5045    | 1.5771
    2 | 0.6703   | 0.7190 | 1.4713    | 1.2616
    3 | 0.7338   | 0.7510 | 1.1829    | 1.1306
    4 | 0.7796   | 0.7780 | 0.9871    | 1.0457
    5 | 0.8074   | 0.8020 | 0.8428    | 0.9689
    6 | 0.8349   | 0.8110 | 0.7220    | 0.9273
    7 | 0.8596   | 0.8140 | 0.6146    | 0.8975
    8 | 0.8794   | 0.8030 | 0.5322    | 0.9035
    9 | 0.8954   | 0.8120 | 0.4592    | 0.8911
   10 | 0.9058   | 0.8200 | 0.4050    | 0.8746
   11 | 0.9172   | 0.8190 | 0.3600    | 0.8713
   12 | 0.9261   | 0.8220 | 0.3115    | 0.8723
   13 | 0.9323   | 0.8150 | 0.2780    | 0.9095
   14 | 0.9354   | 0.8250 | 0.2646    | 0.9278
   15 | 0.9416   | 0.8140 


Results for Dropout Rate = 0.5:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4441   | 0.5880 | 2.5792    | 1.6693
    2 | 0.6086   | 0.6850 | 1.6613    | 1.3838
    3 | 0.6792   | 0.7220 | 1.3855    | 1.2332
    4 | 0.7207   | 0.7370 | 1.2052    | 1.1404
    5 | 0.7444   | 0.7590 | 1.0885    | 1.0772
    6 | 0.7749   | 0.7670 | 0.9594    | 1.0379
    7 | 0.7973   | 0.7840 | 0.8633    | 0.9863
    8 | 0.8089   | 0.8020 | 0.8042    | 0.9582
    9 | 0.8265   | 0.8050 | 0.7253    | 0.9390
   10 | 0.8395   | 0.8100 | 0.6596    | 0.9251
   11 | 0.8519   | 0.8110 | 0.6189    | 0.9293
   12 | 0.8634   | 0.8050 | 0.5563    | 0.9407
   13 | 0.8733   | 0.8170 | 0.5134    | 0.9126
   14 | 0.8790   | 0.8170 | 0.4921    | 0.9153
   15 | 0.8849   | 0.8140 | 0.4585    | 0.9264
   16 | 0.8939   | 0.8120 | 0.4143    | 0.9547
   17 | 0.8988   | 0.8230 | 0.3943    | 0.9365
   18 | 0.9077   | 0.8220 | 0.3792    | 0.9449
   19 | 0.9080   |


Results for Dropout Rate = 0.7:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.2819   | 0.5410 | 3.1799    | 2.0705
    2 | 0.5050   | 0.6030 | 2.1309    | 1.6572
    3 | 0.5690   | 0.6530 | 1.8066    | 1.4973
    4 | 0.5998   | 0.6790 | 1.6571    | 1.4073
    5 | 0.6318   | 0.7020 | 1.5289    | 1.3203
    6 | 0.6538   | 0.7130 | 1.4363    | 1.2735
    7 | 0.6799   | 0.7170 | 1.3529    | 1.2372
    8 | 0.6893   | 0.7230 | 1.2893    | 1.1943
    9 | 0.7072   | 0.7310 | 1.2295    | 1.1787
   10 | 0.7132   | 0.7350 | 1.1704    | 1.1514
   11 | 0.7229   | 0.7530 | 1.1368    | 1.1228
   12 | 0.7316   | 0.7560 | 1.0876    | 1.1114
   13 | 0.7478   | 0.7570 | 1.0273    | 1.1041
   14 | 0.7543   | 0.7610 | 1.0116    | 1.0996
   15 | 0.7591   | 0.7720 | 0.9737    | 1.0872
   16 | 0.7651   | 0.7720 | 0.9470    | 1.0754
   17 | 0.7712   | 0.7760 | 0.9152    | 1.0638
   18 | 0.7791   | 0.7760 | 0.8830    | 1.0616
   19 | 0.7811   |

In [26]:
import matplotlib.pyplot as plt
from tensorflow.keras import models, layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import reuters
import numpy as np

# بارگذاری داده‌ها
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=10000)

# تبدیل به one-hot
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(x_train)
x_test = vectorize_sequences(x_test)

one_hot_train_labels = to_categorical(y_train)
one_hot_test_labels = to_categorical(y_test)

# تقسیم داده‌های آموزش به train و validation
x_val = x_train[:1000]
partial_x_train = x_train[1000:]

y_val = one_hot_train_labels[:1000]
partial_y_train = one_hot_train_labels[1000:]

# تابع برای آموزش و رسم نمودار
def train_and_plot(dropout_rates, neurons, epochs):
    history_dict = {}
    # *** تغییر جدید: ذخیره بهترین مدل‌ها ***
    best_models = []  # لیست برای ذخیره اطلاعات بهترین مدل‌ها
    
    # ساخت مدل
    def build_model(dropout_rate):
        model = models.Sequential()
        model.add(layers.Dense(neurons, activation='relu', input_shape=(10000,)))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(neurons, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])
        return model
    
    # آموزش مدل‌ها
    for rate in dropout_rates:
        print(f"\nTraining with dropout rate = {rate}, neurons = {neurons}")
        model = build_model(rate)
        history = model.fit(
            partial_x_train, partial_y_train,
            epochs=epochs,
            batch_size=512,
            validation_data=(x_val, y_val),
            verbose=0
        )
        history_dict[rate] = history
        
        # *** تغییر جدید: یافتن بهترین val_accuracy برای این نرخ Dropout ***
        history_data = history.history
        max_val_acc = max(history_data['val_accuracy'])
        max_val_acc_epoch = history_data['val_accuracy'].index(max_val_acc) + 1
        best_model_info = {
            'dropout_rate': rate,
            'neurons': neurons,
            'epoch': max_val_acc_epoch,
            'val_accuracy': max_val_acc,
            'val_loss': history_data['val_loss'][max_val_acc_epoch - 1],
            'train_accuracy': history_data['accuracy'][max_val_acc_epoch - 1],
            'train_loss': history_data['loss'][max_val_acc_epoch - 1]
        }
        best_models.append(best_model_info)
    
    # رسم نمودارهای جداگانه برای هر نرخ Dropout
    for rate in dropout_rates:
        history = history_dict[rate].history
        
        # چاپ مقادیر برای هر epoch
        print(f"\nResults for Dropout Rate = {rate}:")
        print("Epoch | Train Acc | Val Acc | Train Loss | Val Loss")
        print("-" * 50)
        for epoch in range(len(history['accuracy'])):
            print(f"{epoch+1:5d} | {history['accuracy'][epoch]:.4f}   | {history['val_accuracy'][epoch]:.4f} | {history['loss'][epoch]:.4f}    | {history['val_loss'][epoch]:.4f}")
        
        # رسم نمودارهای جداگانه
        plt.figure(figsize=(12, 4))
        
        # نمودار دقت
        plt.subplot(1, 2, 1)
        plt.plot(history['accuracy'], label='Train Accuracy', color='blue')
        plt.plot(history['val_accuracy'], '--', label='Validation Accuracy', color='red')
        plt.title(f'Accuracy (Dropout={rate})')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)
        
        # نمودار خطا
        plt.subplot(1, 2, 2)
        plt.plot(history['loss'], label='Train Loss', color='blue')
        plt.plot(history['val_loss'], '--', label='Validation Loss', color='red')
        plt.title(f'Loss (Dropout={rate})')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        
        plt.tight_layout()
        plt.show()
    
    # رسم نمودار مقایسه‌ای برای همه نرخ‌های Dropout
    plt.figure(figsize=(12, 8))
    
    # نمودار دقت اعتبارسنجی
    plt.subplot(2, 1, 1)
    for rate in dropout_rates:
        plt.plot(history_dict[rate].history['val_accuracy'], label=f'Dropout={rate}')
    plt.title(f'Validation Accuracy Comparison (Neurons={neurons})')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Accuracy')
    plt.legend(loc='best')
    plt.grid(True)
    
    # نمودار خطای اعتبارسنجی
    plt.subplot(2, 1, 2)
    for rate in dropout_rates:
        plt.plot(history_dict[rate].history['val_loss'], label=f'Dropout={rate}')
    plt.title(f'Validation Loss Comparison (Neurons={neurons})')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Loss')
    plt.legend(loc='best')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # *** تغییر جدید: چاپ بهترین مدل‌ها برای این تعداد نورون ***
    print(f"\nBest Models for {neurons} Neurons:")
    print("Dropout | Epoch | Train Acc | Val Acc | Train Loss | Val Loss")
    print("-" * 60)
    for model_info in best_models:
        print(f"{model_info['dropout_rate']:.1f}     | {model_info['epoch']:5d} | {model_info['train_accuracy']:.4f}   | {model_info['val_accuracy']:.4f} | {model_info['train_loss']:.4f}    | {model_info['val_loss']:.4f}")

# اجرای آموزش برای مدل با 64 نورون (20 epoch)
print("Training Model with 64 Neurons")
train_and_plot(dropout_rates=[0.3, 0.5, 0.7], neurons=64, epochs=20)

# اجرای آموزش برای مدل با 128 نورون (30 epoch)
print("\nTraining Model with 128 Neurons")
train_and_plot(dropout_rates=[0.3, 0.5, 0.7], neurons=128, epochs=30)

Training Model with 64 Neurons

Training with dropout rate = 0.3, neurons = 64

Training with dropout rate = 0.5, neurons = 64

Training with dropout rate = 0.7, neurons = 64

Results for Dropout Rate = 0.3:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4405   | 0.6080 | 2.7894    | 1.8287
    2 | 0.6146   | 0.6690 | 1.7461    | 1.4220
    3 | 0.6792   | 0.7120 | 1.4269    | 1.2607
    4 | 0.7176   | 0.7310 | 1.2314    | 1.1653
    5 | 0.7417   | 0.7520 | 1.1111    | 1.0951
    6 | 0.7686   | 0.7640 | 1.0045    | 1.0441
    7 | 0.7851   | 0.7750 | 0.9089    | 0.9979
    8 | 0.8044   | 0.7870 | 0.8248    | 0.9770
    9 | 0.8172   | 0.7960 | 0.7652    | 0.9475
   10 | 0.8348   | 0.8100 | 0.6862    | 0.9156
   11 | 0.8448   | 0.8120 | 0.6472    | 0.9102
   12 | 0.8555   | 0.8110 | 0.5950    | 0.9048
   13 | 0.8663   | 0.8170 | 0.5481    | 0.8873
   14 | 0.8779   | 0.8180 | 0.5020    | 0.8915
   15 | 0.8834   | 0.8150 | 0.4


Results for Dropout Rate = 0.5:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.3467   | 0.5140 | 3.0767    | 2.1540
    2 | 0.5061   | 0.6030 | 2.1246    | 1.6706
    3 | 0.5783   | 0.6650 | 1.7964    | 1.4936
    4 | 0.6250   | 0.6900 | 1.5944    | 1.3663
    5 | 0.6572   | 0.7080 | 1.4453    | 1.2744
    6 | 0.6824   | 0.7100 | 1.3499    | 1.2182
    7 | 0.6986   | 0.7180 | 1.2542    | 1.1630
    8 | 0.7124   | 0.7380 | 1.1951    | 1.1304
    9 | 0.7255   | 0.7490 | 1.1127    | 1.1028
   10 | 0.7392   | 0.7540 | 1.0580    | 1.0801
   11 | 0.7511   | 0.7680 | 1.0133    | 1.0493
   12 | 0.7637   | 0.7680 | 0.9688    | 1.0343
   13 | 0.7755   | 0.7740 | 0.9207    | 1.0127
   14 | 0.7816   | 0.7740 | 0.8771    | 1.0220
   15 | 0.7895   | 0.7820 | 0.8446    | 0.9996
   16 | 0.7962   | 0.7830 | 0.8122    | 1.0049
   17 | 0.8037   | 0.7870 | 0.7746    | 1.0027
   18 | 0.8077   | 0.7900 | 0.7629    | 0.9795
   19 | 0.8168   |


Results for Dropout Rate = 0.7:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.2042   | 0.5350 | 3.3940    | 2.5533
    2 | 0.3988   | 0.5770 | 2.5630    | 1.8783
    3 | 0.4713   | 0.5980 | 2.1933    | 1.6393
    4 | 0.5148   | 0.6240 | 1.9682    | 1.5546
    5 | 0.5470   | 0.6440 | 1.8355    | 1.4988
    6 | 0.5593   | 0.6780 | 1.7504    | 1.4678
    7 | 0.5799   | 0.6830 | 1.6705    | 1.4198
    8 | 0.5945   | 0.6960 | 1.5921    | 1.3886
    9 | 0.6031   | 0.7040 | 1.5611    | 1.3594
   10 | 0.6193   | 0.7040 | 1.5215    | 1.3379
   11 | 0.6289   | 0.7170 | 1.4727    | 1.3143
   12 | 0.6381   | 0.7150 | 1.4325    | 1.2933
   13 | 0.6416   | 0.7160 | 1.4075    | 1.2872
   14 | 0.6505   | 0.7160 | 1.3715    | 1.2702
   15 | 0.6671   | 0.7190 | 1.3243    | 1.2582
   16 | 0.6625   | 0.7230 | 1.3220    | 1.2536
   17 | 0.6729   | 0.7210 | 1.2790    | 1.2446
   18 | 0.6723   | 0.7220 | 1.2673    | 1.2362
   19 | 0.6850   |


Best Models for 64 Neurons:
Dropout | Epoch | Train Acc | Val Acc | Train Loss | Val Loss
------------------------------------------------------------
0.3     |    16 | 0.8950   | 0.8210 | 0.4305    | 0.8950
0.5     |    20 | 0.8207   | 0.7930 | 0.7076    | 0.9949
0.7     |    19 | 0.6850   | 0.7250 | 1.2358    | 1.2385

Training Model with 128 Neurons

Training with dropout rate = 0.3, neurons = 128

Training with dropout rate = 0.5, neurons = 128

Training with dropout rate = 0.7, neurons = 128

Results for Dropout Rate = 0.3:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4744   | 0.6300 | 2.5167    | 1.5637
    2 | 0.6659   | 0.7100 | 1.4764    | 1.2550
    3 | 0.7358   | 0.7430 | 1.1718    | 1.0957
    4 | 0.7780   | 0.7910 | 0.9789    | 0.9925
    5 | 0.8183   | 0.7830 | 0.8230    | 0.9633
    6 | 0.8396   | 0.7990 | 0.6989    | 0.9109
    7 | 0.8631   | 0.8170 | 0.5990    | 0.8657
    8 | 0.8796   | 0.8210 | 0.51


Results for Dropout Rate = 0.5:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4308   | 0.6130 | 2.6488    | 1.6574
    2 | 0.6179   | 0.6910 | 1.6562    | 1.3666
    3 | 0.6840   | 0.7160 | 1.3832    | 1.2559
    4 | 0.7120   | 0.7350 | 1.2356    | 1.1629
    5 | 0.7414   | 0.7580 | 1.1095    | 1.0926
    6 | 0.7655   | 0.7700 | 1.0050    | 1.0404
    7 | 0.7888   | 0.7830 | 0.9064    | 0.9970
    8 | 0.8018   | 0.7920 | 0.8289    | 0.9614
    9 | 0.8145   | 0.7970 | 0.7610    | 0.9465
   10 | 0.8365   | 0.8010 | 0.6888    | 0.9350
   11 | 0.8415   | 0.8150 | 0.6388    | 0.9090
   12 | 0.8559   | 0.8050 | 0.5817    | 0.9132
   13 | 0.8653   | 0.8190 | 0.5483    | 0.8996
   14 | 0.8766   | 0.8230 | 0.5012    | 0.9133
   15 | 0.8867   | 0.8180 | 0.4686    | 0.9058
   16 | 0.8882   | 0.8170 | 0.4326    | 0.9233
   17 | 0.8954   | 0.8160 | 0.4121    | 0.9289
   18 | 0.9020   | 0.8230 | 0.3792    | 0.9254
   19 | 0.9043   |


Results for Dropout Rate = 0.7:
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.3104   | 0.5380 | 3.0282    | 1.9460
    2 | 0.4995   | 0.6150 | 2.0794    | 1.6228
    3 | 0.5664   | 0.6660 | 1.8029    | 1.5083
    4 | 0.6161   | 0.6930 | 1.6339    | 1.3964
    5 | 0.6407   | 0.6980 | 1.5258    | 1.3293
    6 | 0.6607   | 0.7070 | 1.4497    | 1.2718
    7 | 0.6728   | 0.7180 | 1.3711    | 1.2266
    8 | 0.6889   | 0.7240 | 1.2941    | 1.1969
    9 | 0.7012   | 0.7300 | 1.2348    | 1.1707
   10 | 0.7067   | 0.7350 | 1.1865    | 1.1584
   11 | 0.7216   | 0.7440 | 1.1396    | 1.1375
   12 | 0.7314   | 0.7500 | 1.1113    | 1.1170
   13 | 0.7415   | 0.7570 | 1.0673    | 1.1078
   14 | 0.7498   | 0.7600 | 1.0168    | 1.0911
   15 | 0.7522   | 0.7660 | 0.9794    | 1.0836
   16 | 0.7628   | 0.7670 | 0.9492    | 1.0806
   17 | 0.7674   | 0.7740 | 0.9252    | 1.0675
   18 | 0.7766   | 0.7750 | 0.8949    | 1.0750
   19 | 0.7774   |


Best Models for 128 Neurons:
Dropout | Epoch | Train Acc | Val Acc | Train Loss | Val Loss
------------------------------------------------------------
0.3     |    11 | 0.9188   | 0.8320 | 0.3505    | 0.8474
0.5     |    30 | 0.9385   | 0.8310 | 0.2272    | 1.0286
0.7     |    29 | 0.8167   | 0.7920 | 0.6763    | 1.0794


In [27]:
import matplotlib.pyplot as plt
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import reuters
import numpy as np

# بارگذاری داده‌ها
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=10000)

# تبدیل به one-hot
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(x_train)
x_test = vectorize_sequences(x_test)

one_hot_train_labels = to_categorical(y_train)
one_hot_test_labels = to_categorical(y_test)

# تقسیم داده‌های آموزش به train و validation
x_val = x_train[:1000]
partial_x_train = x_train[1000:]

y_val = one_hot_train_labels[:1000]
partial_y_train = one_hot_train_labels[1000:]

# تابع برای آموزش و رسم نمودار با منظم‌سازی‌های مختلف
def train_and_plot(regularization_configs, neurons, epochs):
    history_dict = {}
    best_models = []  # ذخیره بهترین مدل‌ها
    
    # ساخت مدل
    def build_model(reg_type, reg_param):
        model = models.Sequential()
        if reg_type == 'dropout':
            model.add(layers.Dense(neurons, activation='relu', input_shape=(10000,)))
            model.add(layers.Dropout(reg_param))
            model.add(layers.Dense(neurons, activation='relu'))
            model.add(layers.Dropout(reg_param))
        elif reg_type == 'l1':
            model.add(layers.Dense(neurons, activation='relu', input_shape=(10000,),
                                  kernel_regularizer=regularizers.l1(reg_param)))
            model.add(layers.Dense(neurons, activation='relu',
                                  kernel_regularizer=regularizers.l1(reg_param)))
        elif reg_type == 'l2':
            model.add(layers.Dense(neurons, activation='relu', input_shape=(10000,),
                                  kernel_regularizer=regularizers.l2(reg_param)))
            model.add(layers.Dense(neurons, activation='relu',
                                  kernel_regularizer=regularizers.l2(reg_param)))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])
        return model
    
    # آموزش مدل‌ها
    for reg_type, reg_param in regularization_configs:
        print(f"\nTraining with {reg_type} (param={reg_param}), neurons={neurons}")
        model = build_model(reg_type, reg_param)
        history = model.fit(
            partial_x_train, partial_y_train,
            epochs=epochs,
            batch_size=512,
            validation_data=(x_val, y_val),
            verbose=0
        )
        history_dict[(reg_type, reg_param)] = history
        
        # یافتن بهترین val_accuracy
        history_data = history.history
        max_val_acc = max(history_data['val_accuracy'])
        max_val_acc_epoch = history_data['val_accuracy'].index(max_val_acc) + 1
        best_model_info = {
            'reg_type': reg_type,
            'reg_param': reg_param,
            'neurons': neurons,
            'epoch': max_val_acc_epoch,
            'val_accuracy': max_val_acc,
            'val_loss': history_data['val_loss'][max_val_acc_epoch - 1],
            'train_accuracy': history_data['accuracy'][max_val_acc_epoch - 1],
            'train_loss': history_data['loss'][max_val_acc_epoch - 1]
        }
        best_models.append(best_model_info)
    
    # رسم نمودارهای جداگانه برای هر نوع منظم‌سازی
    for reg_type, reg_param in regularization_configs:
        history = history_dict[(reg_type, reg_param)].history
        
        print(f"\nResults for {reg_type} (param={reg_param}):")
        print("Epoch | Train Acc | Val Acc | Train Loss | Val Loss")
        print("-" * 50)
        for epoch in range(len(history['accuracy'])):
            print(f"{epoch+1:5d} | {history['accuracy'][epoch]:.4f}   | {history['val_accuracy'][epoch]:.4f} | {history['loss'][epoch]:.4f}    | {history['val_loss'][epoch]:.4f}")
        
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(history['accuracy'], label='Train Accuracy', color='blue')
        plt.plot(history['val_accuracy'], '--', label='Validation Accuracy', color='red')
        plt.title(f'Accuracy ({reg_type}={reg_param})')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)
        
        plt.subplot(1, 2, 2)
        plt.plot(history['loss'], label='Train Loss', color='blue')
        plt.plot(history['val_loss'], '--', label='Validation Loss', color='red')
        plt.title(f'Loss ({reg_type}={reg_param})')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(f'{reg_type}_{reg_param}_neurons_{neurons}.png')
        plt.show()
    
    # رسم نمودار مقایسه‌ای
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 1, 1)
    for reg_type, reg_param in regularization_configs:
        plt.plot(history_dict[(reg_type, reg_param)].history['val_accuracy'],
                 label=f'{reg_type}={reg_param}')
    plt.title(f'Validation Accuracy Comparison (Neurons={neurons})')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Accuracy')
    plt.legend(loc='best')
    plt.grid(True)
    
    plt.subplot(2, 1, 2)
    for reg_type, reg_param in regularization_configs:
        plt.plot(history_dict[(reg_type, reg_param)].history['val_loss'],
                 label=f'{reg_type}={reg_param}')
    plt.title(f'Validation Loss Comparison (Neurons={neurons})')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Loss')
    plt.legend(loc='best')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(f'comparison_neurons_{neurons}.png')
    plt.show()
    
    # چاپ بهترین مدل‌ها
    print(f"\nBest Models for {neurons} Neurons:")
    print("Reg Type | Param | Epoch | Train Acc | Val Acc | Train Loss | Val Loss")
    print("-" * 70)
    for model_info in best_models:
        print(f"{model_info['reg_type']:<9} | {model_info['reg_param']:.3f} | {model_info['epoch']:5d} | "
              f"{model_info['train_accuracy']:.4f}   | {model_info['val_accuracy']:.4f} | "
              f"{model_info['train_loss']:.4f}    | {model_info['val_loss']:.4f}")

# تنظیمات منظم‌سازی
regularization_configs = [
    ('dropout', 0.3),
    ('dropout', 0.5),
    ('dropout', 0.7),
    ('l1', 0.001),
    ('l2', 0.001)
]

# اجرای آموزش برای مدل با 64 نورون (20 epoch)
print("Training Model with 64 Neurons")
train_and_plot(regularization_configs, neurons=64, epochs=20)

# اجرای آموزش برای مدل با 128 نورون (30 epoch)
print("\nTraining Model with 128 Neurons")
train_and_plot(regularization_configs, neurons=128, epochs=30)

Training Model with 64 Neurons

Training with dropout (param=0.3), neurons=64

Training with dropout (param=0.5), neurons=64

Training with dropout (param=0.7), neurons=64

Training with l1 (param=0.001), neurons=64

Training with l2 (param=0.001), neurons=64

Results for dropout (param=0.3):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.3924   | 0.5820 | 2.8260    | 1.8991
    2 | 0.5884   | 0.6600 | 1.7884    | 1.4634
    3 | 0.6685   | 0.6990 | 1.4469    | 1.2880
    4 | 0.6983   | 0.7230 | 1.2767    | 1.1761
    5 | 0.7291   | 0.7400 | 1.1488    | 1.1000
    6 | 0.7543   | 0.7570 | 1.0367    | 1.0435
    7 | 0.7745   | 0.7760 | 0.9368    | 0.9999
    8 | 0.7969   | 0.7880 | 0.8584    | 0.9570
    9 | 0.8155   | 0.7900 | 0.7795    | 0.9289
   10 | 0.8259   | 0.8010 | 0.7149    | 0.9117
   11 | 0.8429   | 0.8060 | 0.6623    | 0.8983
   12 | 0.8510   | 0.8090 | 0.6105    | 0.8815
   13 | 0.8671   | 0.8080 | 0.5591    |


Results for dropout (param=0.5):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.3269   | 0.5500 | 3.1180    | 2.1272
    2 | 0.5289   | 0.6360 | 2.1143    | 1.6318
    3 | 0.5986   | 0.6730 | 1.7441    | 1.4627
    4 | 0.6433   | 0.6910 | 1.5529    | 1.3418
    5 | 0.6681   | 0.7070 | 1.4381    | 1.2792
    6 | 0.6906   | 0.7170 | 1.3345    | 1.2126
    7 | 0.7018   | 0.7290 | 1.2464    | 1.1744
    8 | 0.7147   | 0.7360 | 1.1859    | 1.1425
    9 | 0.7304   | 0.7400 | 1.1101    | 1.1139
   10 | 0.7440   | 0.7520 | 1.0641    | 1.0812
   11 | 0.7549   | 0.7600 | 1.0075    | 1.0643
   12 | 0.7647   | 0.7630 | 0.9603    | 1.0505
   13 | 0.7696   | 0.7770 | 0.9263    | 1.0369
   14 | 0.7814   | 0.7760 | 0.8847    | 1.0300
   15 | 0.7935   | 0.7780 | 0.8366    | 1.0290
   16 | 0.7905   | 0.7770 | 0.8217    | 1.0182
   17 | 0.7989   | 0.7800 | 0.7982    | 1.0198
   18 | 0.8057   | 0.7870 | 0.7583    | 1.0217
   19 | 0.8128   


Results for dropout (param=0.7):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.2503   | 0.5350 | 3.2275    | 2.3738
    2 | 0.4171   | 0.5480 | 2.4816    | 1.8716
    3 | 0.4716   | 0.5710 | 2.1337    | 1.6834
    4 | 0.5081   | 0.6050 | 1.9658    | 1.5834
    5 | 0.5387   | 0.6370 | 1.8378    | 1.5274
    6 | 0.5664   | 0.6630 | 1.7430    | 1.4803
    7 | 0.5838   | 0.6720 | 1.6771    | 1.4299
    8 | 0.5884   | 0.6860 | 1.6152    | 1.3914
    9 | 0.6065   | 0.6960 | 1.5660    | 1.3555
   10 | 0.6218   | 0.6950 | 1.5165    | 1.3261
   11 | 0.6304   | 0.7070 | 1.4708    | 1.3093
   12 | 0.6378   | 0.7070 | 1.4403    | 1.2881
   13 | 0.6437   | 0.7150 | 1.4063    | 1.2720
   14 | 0.6540   | 0.7160 | 1.3796    | 1.2555
   15 | 0.6571   | 0.7130 | 1.3374    | 1.2483
   16 | 0.6645   | 0.7190 | 1.3112    | 1.2386
   17 | 0.6750   | 0.7230 | 1.2832    | 1.2270
   18 | 0.6817   | 0.7200 | 1.2584    | 1.2258
   19 | 0.6869   


Results for l1 (param=0.001):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4346   | 0.5500 | 6.4986    | 3.2489
    2 | 0.5798   | 0.6060 | 2.8725    | 2.6446
    3 | 0.6424   | 0.6650 | 2.5541    | 2.4511
    4 | 0.6719   | 0.6670 | 2.4057    | 2.3720
    5 | 0.6823   | 0.6890 | 2.3061    | 2.2719
    6 | 0.6929   | 0.6960 | 2.2249    | 2.1872
    7 | 0.6968   | 0.6780 | 2.1658    | 2.2025
    8 | 0.7040   | 0.6980 | 2.1198    | 2.1011
    9 | 0.7092   | 0.7090 | 2.0676    | 2.0738
   10 | 0.7164   | 0.7120 | 2.0282    | 2.0147
   11 | 0.7190   | 0.7150 | 1.9990    | 2.0017
   12 | 0.7244   | 0.7150 | 1.9626    | 1.9775
   13 | 0.7290   | 0.7230 | 1.9438    | 1.9471
   14 | 0.7339   | 0.7240 | 1.9155    | 1.9319
   15 | 0.7372   | 0.7340 | 1.8952    | 1.8982
   16 | 0.7409   | 0.7310 | 1.8717    | 1.9044
   17 | 0.7451   | 0.7300 | 1.8553    | 1.9144
   18 | 0.7483   | 0.7580 | 1.8393    | 1.8656
   19 | 0.7568   | 0


Results for l2 (param=0.001):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4905   | 0.5940 | 2.9590    | 2.0671
    2 | 0.6538   | 0.6700 | 1.8032    | 1.6175
    3 | 0.7266   | 0.7210 | 1.4413    | 1.4127
    4 | 0.7682   | 0.7400 | 1.2246    | 1.3241
    5 | 0.8048   | 0.7720 | 1.0716    | 1.2271
    6 | 0.8317   | 0.8030 | 0.9505    | 1.1567
    7 | 0.8550   | 0.8010 | 0.8585    | 1.1391
    8 | 0.8763   | 0.8140 | 0.7860    | 1.1102
    9 | 0.8950   | 0.8120 | 0.7145    | 1.0981
   10 | 0.9043   | 0.8190 | 0.6700    | 1.0684
   11 | 0.9157   | 0.8220 | 0.6220    | 1.0769
   12 | 0.9219   | 0.8110 | 0.5946    | 1.0829
   13 | 0.9303   | 0.8340 | 0.5554    | 1.0586
   14 | 0.9344   | 0.8240 | 0.5341    | 1.0738
   15 | 0.9377   | 0.8220 | 0.5180    | 1.0702
   16 | 0.9431   | 0.8070 | 0.4966    | 1.1007
   17 | 0.9458   | 0.8170 | 0.4777    | 1.0644
   18 | 0.9488   | 0.8190 | 0.4692    | 1.0772
   19 | 0.9495   | 0

...


Results for dropout (param=0.5):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4316   | 0.6000 | 2.6380    | 1.6678
    2 | 0.6074   | 0.6860 | 1.6793    | 1.3688
    3 | 0.6700   | 0.7080 | 1.4189    | 1.2373
    4 | 0.7066   | 0.7310 | 1.2483    | 1.1394
    5 | 0.7347   | 0.7510 | 1.1280    | 1.0769
    6 | 0.7636   | 0.7680 | 1.0141    | 1.0298
    7 | 0.7747   | 0.7830 | 0.9429    | 0.9994
    8 | 0.7997   | 0.7940 | 0.8348    | 0.9693
    9 | 0.8126   | 0.8040 | 0.7716    | 0.9439
   10 | 0.8299   | 0.8050 | 0.6981    | 0.9414
   11 | 0.8343   | 0.8060 | 0.6621    | 0.9131
   12 | 0.8514   | 0.8160 | 0.5885    | 0.9127
   13 | 0.8627   | 0.8210 | 0.5504    | 0.9106
   14 | 0.8712   | 0.8230 | 0.5145    | 0.8998
   15 | 0.8839   | 0.8220 | 0.4726    | 0.9140
   16 | 0.8886   | 0.8260 | 0.4491    | 0.8906
   17 | 0.8959   | 0.8240 | 0.4132    | 0.9003
   18 | 0.8998   | 0.8210 | 0.3916    | 0.9394
   19 | 0.9069   


Results for dropout (param=0.7):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.2777   | 0.5410 | 3.1516    | 2.0466
    2 | 0.5015   | 0.6010 | 2.1085    | 1.6448
    3 | 0.5633   | 0.6550 | 1.8202    | 1.4935
    4 | 0.5991   | 0.6890 | 1.6520    | 1.3988
    5 | 0.6337   | 0.7000 | 1.5355    | 1.3276
    6 | 0.6611   | 0.7060 | 1.4416    | 1.2637
    7 | 0.6822   | 0.7130 | 1.3348    | 1.2272
    8 | 0.6913   | 0.7210 | 1.2923    | 1.1975
    9 | 0.7060   | 0.7310 | 1.2375    | 1.1694
   10 | 0.7139   | 0.7300 | 1.1783    | 1.1560
   11 | 0.7239   | 0.7420 | 1.1351    | 1.1353
   12 | 0.7315   | 0.7510 | 1.0931    | 1.1216
   13 | 0.7425   | 0.7560 | 1.0550    | 1.1223
   14 | 0.7496   | 0.7610 | 1.0157    | 1.1121
   15 | 0.7596   | 0.7680 | 0.9746    | 1.1003
   16 | 0.7704   | 0.7710 | 0.9522    | 1.0886
   17 | 0.7700   | 0.7710 | 0.9202    | 1.0875
   18 | 0.7770   | 0.7750 | 0.8814    | 1.0969
   19 | 0.7789   


Results for l1 (param=0.001):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4607   | 0.5720 | 9.8645    | 4.0646
    2 | 0.6144   | 0.6710 | 3.6700    | 3.4184
    3 | 0.6596   | 0.6750 | 3.3316    | 3.1914
    4 | 0.6817   | 0.6840 | 3.1075    | 3.0222
    5 | 0.6849   | 0.6930 | 2.9509    | 2.8701
    6 | 0.6906   | 0.6940 | 2.8187    | 2.7399
    7 | 0.6981   | 0.6870 | 2.6974    | 2.6660
    8 | 0.7023   | 0.6830 | 2.6156    | 2.6143
    9 | 0.7047   | 0.7050 | 2.5430    | 2.5058
   10 | 0.7119   | 0.7090 | 2.4740    | 2.4894
   11 | 0.7221   | 0.7260 | 2.4231    | 2.4353
   12 | 0.7261   | 0.7130 | 2.3925    | 2.3846
   13 | 0.7315   | 0.7320 | 2.3424    | 2.3750
   14 | 0.7380   | 0.7290 | 2.3108    | 2.3405
   15 | 0.7438   | 0.7270 | 2.2746    | 2.2895
   16 | 0.7449   | 0.7400 | 2.2547    | 2.2964
   17 | 0.7521   | 0.7150 | 2.2112    | 2.2940
   18 | 0.7547   | 0.7480 | 2.1992    | 2.2254
   19 | 0.7586   | 0


Results for l2 (param=0.001):
Epoch | Train Acc | Val Acc | Train Loss | Val Loss
--------------------------------------------------
    1 | 0.4900   | 0.6580 | 2.7464    | 1.9119
    2 | 0.7096   | 0.7170 | 1.6737    | 1.5897
    3 | 0.7716   | 0.7370 | 1.3588    | 1.4245
    4 | 0.8150   | 0.7870 | 1.1461    | 1.2855
    5 | 0.8494   | 0.7790 | 0.9899    | 1.2639
    6 | 0.8768   | 0.8030 | 0.8728    | 1.1838
    7 | 0.8939   | 0.8030 | 0.7938    | 1.1757
    8 | 0.9143   | 0.7880 | 0.7020    | 1.1995
    9 | 0.9221   | 0.8160 | 0.6645    | 1.1482
   10 | 0.9287   | 0.8180 | 0.6235    | 1.1172
   11 | 0.9311   | 0.8180 | 0.5987    | 1.1179
   12 | 0.9441   | 0.8000 | 0.5468    | 1.1574
   13 | 0.9456   | 0.7950 | 0.5356    | 1.2314
   14 | 0.9434   | 0.8190 | 0.5198    | 1.1163
   15 | 0.9505   | 0.7920 | 0.4975    | 1.2243
   16 | 0.9484   | 0.7920 | 0.4824    | 1.1666
   17 | 0.9471   | 0.8110 | 0.4905    | 1.1300
   18 | 0.9524   | 0.8130 | 0.4596    | 1.1153
   19 | 0.9518   | 0


Best Models for 128 Neurons:
Reg Type | Param | Epoch | Train Acc | Val Acc | Train Loss | Val Loss
----------------------------------------------------------------------
dropout   | 0.300 |    11 | 0.9208   | 0.8240 | 0.3406    | 0.8530
dropout   | 0.500 |    27 | 0.9320   | 0.8300 | 0.2550    | 0.9720
dropout   | 0.700 |    30 | 0.8256   | 0.8050 | 0.6590    | 1.1139
l1        | 0.001 |    29 | 0.7818   | 0.7790 | 1.9848    | 2.0471
l2        | 0.001 |    14 | 0.9434   | 0.8190 | 0.5198    | 1.1163


In [11]:
import numpy as np
from keras import models, layers, regularizers
from keras.datasets import reuters
from keras.utils import to_categorical
import matplotlib.pyplot as plt

# بارگذاری دیتاست
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

# تبدیل داده‌ها به بردار
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)

# داده‌های اعتبارسنجی
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = y_train[:1000]
partial_y_train = y_train[1000:]

# پارامترهای آزمایشی
dropout_rates = [0.3, 0.5, 0.7]
l1_values = [0.0001, 0.001, 0.01]
l2_values = [0.0002,0.0001, 0.001, 0.01]

results = {}

# تابع آموزش و ارزیابی مدل
def train_model(name, model):
    print(f"Training: {name}")
    history = model.fit(partial_x_train, partial_y_train,
                        epochs=30, batch_size=512,
                        validation_data=(x_val, y_val),
                        verbose=0)
    val_acc = history.history['val_accuracy'][-1]
    results[name] = val_acc
    return val_acc

# 🟦 فقط L2 Regularization
for val in l2_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L2={val}", model)

# 🟩 فقط L1 Regularization
for val in l1_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L1={val}", model)

# 🟨 ترکیب L1 و L2
for l1_val in l1_values:
    for l2_val in l2_values:
        reg = regularizers.l1_l2(l1=l1_val, l2=l2_val)
        model = models.Sequential()
        model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=reg))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=reg))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"L1={l1_val}_L2={l2_val}", model)

# 🟧 فقط Dropout
for rate in dropout_rates:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,)))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"Dropout={rate}", model)

# 🟥 Dropout + L2 Regularization
for rate in dropout_rates:
    for l2_val in l2_values:
        model = models.Sequential()
        model.add(layers.Dense(256, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"Dropout={rate}_L2={l2_val}", model)

# نمایش بهترین مدل
best_model_name = max(results, key=results.get)
best_val_acc = results[best_model_name]

print("\nBest Model:")
print(f"{best_model_name} => Validation Accuracy: {best_val_acc:.4f}")

# چاپ همه‌ی نتایج
print("\nAll Results:")
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {acc:.4f}")


Training: L2=0.0002
Training: L2=0.0001
Training: L2=0.001
Training: L2=0.01
Training: L1=0.0001
Training: L1=0.001
Training: L1=0.01
Training: L1=0.0001_L2=0.0002
Training: L1=0.0001_L2=0.0001
Training: L1=0.0001_L2=0.001
Training: L1=0.0001_L2=0.01
Training: L1=0.001_L2=0.0002
Training: L1=0.001_L2=0.0001
Training: L1=0.001_L2=0.001
Training: L1=0.001_L2=0.01
Training: L1=0.01_L2=0.0002
Training: L1=0.01_L2=0.0001
Training: L1=0.01_L2=0.001
Training: L1=0.01_L2=0.01
Training: Dropout=0.3
Training: Dropout=0.5
Training: Dropout=0.7
Training: Dropout=0.3_L2=0.0002
Training: Dropout=0.3_L2=0.0001
Training: Dropout=0.3_L2=0.001
Training: Dropout=0.3_L2=0.01
Training: Dropout=0.5_L2=0.0002
Training: Dropout=0.5_L2=0.0001
Training: Dropout=0.5_L2=0.001
Training: Dropout=0.5_L2=0.01
Training: Dropout=0.7_L2=0.0002
Training: Dropout=0.7_L2=0.0001
Training: Dropout=0.7_L2=0.001
Training: Dropout=0.7_L2=0.01

Best Model:
Dropout=0.5 => Validation Accuracy: 0.8270

All Results:
Dropout=0.5: 0.8

In [2]:
import numpy as np
from keras import models, layers, regularizers
from keras.datasets import reuters
from keras.utils import to_categorical
import matplotlib.pyplot as plt

# بارگذاری دیتاست
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

# تبدیل داده‌ها به بردار
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)

# داده‌های اعتبارسنجی
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = y_train[:1000]
partial_y_train = y_train[1000:]

# پارامترهای آزمایشی
dropout_rates = [0.3, 0.5, 0.7]
l1_values = [0.0001, 0.001, 0.01]
l2_values = [0.0001, 0.001, 0.01]

results = {}

# تابع آموزش و ارزیابی مدل
def train_model(name, model):
    print(f"Training: {name}")
    history = model.fit(partial_x_train, partial_y_train,
                        epochs=30, batch_size=512,
                        validation_data=(x_val, y_val),
                        verbose=0)
    
    # ذخیره نتیجه
    val_acc = history.history['val_accuracy'][-1]
    results[name] = val_acc

    # 🎯 رسم نمودار برای هر مدل
    acc = history.history['accuracy']
    val_acc_list = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(10, 4))
    plt.suptitle(f"{name}", fontsize=14)

    # نمودار دقت
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, color='navy', marker='o', label='Training Accuracy')
    plt.plot(epochs, val_acc_list, color='pink', marker='o', label='Validation Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # نمودار خطا
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, color='navy', marker='o', label='Training Loss')
    plt.plot(epochs, val_loss, color='pink', marker='o', label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    return val_acc


# 🟦 فقط L2 Regularization
for val in l2_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L2={val}", model)

# 🟩 فقط L1 Regularization
for val in l1_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L1={val}", model)

# 🟨 ترکیب L1 و L2
for l1_val in l1_values:
    for l2_val in l2_values:
        reg = regularizers.l1_l2(l1=l1_val, l2=l2_val)
        model = models.Sequential()
        model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=reg))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=reg))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"L1={l1_val}_L2={l2_val}", model)

# 🟧 فقط Dropout
for rate in dropout_rates:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,)))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"Dropout={rate}", model)

# 🟥 Dropout + L2 Regularization
for rate in dropout_rates:
    for l2_val in l2_values:
        model = models.Sequential()
        model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"Dropout={rate}_L2={l2_val}", model)

# نمایش بهترین مدل
best_model_name = max(results, key=results.get)
best_val_acc = results[best_model_name]

print("\nBest Model:")
print(f"{best_model_name} => Validation Accuracy: {best_val_acc:.4f}")

# چاپ همه‌ی نتایج
print("\nAll Results:")
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {acc:.4f}")


Training: L2=0.0001


Training: L2=0.001


Training: L2=0.01


Training: L1=0.0001


Training: L1=0.001


...

Training: Dropout=0.5_L2=0.01


Training: Dropout=0.7_L2=0.0001


Training: Dropout=0.7_L2=0.001


Training: Dropout=0.7_L2=0.01



Best Model:
Dropout=0.5 => Validation Accuracy: 0.8250

All Results:
Dropout=0.5: 0.8250
Dropout=0.3_L2=0.001: 0.8230
Dropout=0.5_L2=0.0001: 0.8220
Dropout=0.5_L2=0.001: 0.8180
Dropout=0.3: 0.8130
Dropout=0.3_L2=0.0001: 0.8110
L2=0.0001: 0.8060
L2=0.001: 0.8040
L1=0.0001: 0.8030
L1=0.0001_L2=0.0001: 0.8010
L1=0.0001_L2=0.001: 0.7980
Dropout=0.7: 0.7980
Dropout=0.7_L2=0.0001: 0.7950
Dropout=0.7_L2=0.001: 0.7850
L2=0.01: 0.7770
Dropout=0.5_L2=0.01: 0.7760
L1=0.0001_L2=0.01: 0.7620
L1=0.001_L2=0.001: 0.7610
L1=0.001: 0.7570
L1=0.001_L2=0.0001: 0.7530
Dropout=0.3_L2=0.01: 0.7520
Dropout=0.7_L2=0.01: 0.7300
L1=0.001_L2=0.01: 0.7150
L1=0.01_L2=0.001: 0.5840
L1=0.01_L2=0.0001: 0.5800
L1=0.01: 0.5770
L1=0.01_L2=0.01: 0.5770


In [ ]:
import numpy as np
from keras import models, layers, regularizers
from keras.datasets import reuters
from keras.utils import to_categorical
import matplotlib.pyplot as plt
import pandas as pd

# بارگذاری دیتاست
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

# تبدیل داده‌ها به بردار
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)

# داده‌های اعتبارسنجی
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = y_train[:1000]
partial_y_train = y_train[1000:]

# پارامترهای آزمایشی
dropout_rates = [0.3, 0.5, 0.7]
l1_values = [0.0001, 0.001, 0.01]
l2_values = [0.0001, 0.001, 0.01]

results = {}

# دیکشنری برای ذخیره مقادیر
metrics = {
    'Model': [],
    'Train Loss': [],
    'Val Loss': [],
    'Train Accuracy': [],
    'Val Accuracy': []
}

# تابع آموزش و ارزیابی مدل با ذخیره مقادیر
def train_model(name, model):
    print(f"Training: {name}")
    history = model.fit(partial_x_train, partial_y_train,
                        epochs=30, batch_size=512,
                        validation_data=(x_val, y_val),
                        verbose=0)
    
    # ذخیره نتایج دقت و خطا
    val_acc = history.history['val_accuracy'][-1]
    results[name] = val_acc
    
    metrics['Model'].append(name)
    metrics['Train Loss'].append(history.history['loss'][-1])
    metrics['Val Loss'].append(history.history['val_loss'][-1])
    metrics['Train Accuracy'].append(history.history['accuracy'][-1])
    metrics['Val Accuracy'].append(history.history['val_accuracy'][-1])

    return val_acc

# 🟦 فقط L2 Regularization
for val in l2_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L2={val}", model)

# 🟩 فقط L1 Regularization
for val in l1_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L1={val}", model)

# 🟨 ترکیب L1 و L2
for l1_val in l1_values:
    for l2_val in l2_values:
        reg = regularizers.l1_l2(l1=l1_val, l2=l2_val)
        model = models.Sequential()
        model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=reg))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=reg))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"L1={l1_val}_L2={l2_val}", model)

# 🟧 فقط Dropout
for rate in dropout_rates:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,)))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"Dropout={rate}", model)

# 🟥 Dropout + L2 Regularization
for rate in dropout_rates:
    for l2_val in l2_values:
        model = models.Sequential()
        model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"Dropout={rate}_L2={l2_val}", model)

# ذخیره کردن نتایج در یک DataFrame برای نمایش
df = pd.DataFrame(metrics)

# نمایش نتایج در یک جدول (با مرتب کردن بر اساس دقت اعتبارسنجی)
# df.sort_values(by='Val Accuracy', ascending=False, inplace=True)

# نمایش جدول
df


Training: L2=0.0001
Training: L2=0.001
Training: L2=0.01
Training: L1=0.0001
Training: L1=0.001
Training: L1=0.01
Training: L1=0.0001_L2=0.0001
Training: L1=0.0001_L2=0.001
Training: L1=0.0001_L2=0.01
Training: L1=0.001_L2=0.0001
Training: L1=0.001_L2=0.001
Training: L1=0.001_L2=0.01
Training: L1=0.01_L2=0.0001
Training: L1=0.01_L2=0.001
Training: L1=0.01_L2=0.01
Training: Dropout=0.3
Training: Dropout=0.5
Training: Dropout=0.7
Training: Dropout=0.3_L2=0.0001
Training: Dropout=0.3_L2=0.001
Training: Dropout=0.3_L2=0.01
Training: Dropout=0.5_L2=0.0001
Training: Dropout=0.5_L2=0.001
Training: Dropout=0.5_L2=0.01
Training: Dropout=0.7_L2=0.0001
Training: Dropout=0.7_L2=0.001
Training: Dropout=0.7_L2=0.01


,Model,Train Loss,Val Loss,Train Accuracy,Val Accuracy
18,Dropout=0.3_L2=0.0001,0.201459,1.029228,0.959659,0.828
16,Dropout=0.5,0.226047,1.045498,0.940867,0.828
21,Dropout=0.5_L2=0.0001,0.305849,1.072247,0.936733,0.822
22,Dropout=0.5_L2=0.001,0.595067,1.122800,0.917063,0.818
6,L1=0.0001_L2=0.0001,0.641006,1.327997,0.945001,0.814
7,L1=0.0001_L2=0.001,0.831243,1.346330,0.928589,0.812
15,Dropout=0.3,0.125279,1.019586,0.961413,0.811
0,L2=0.0001,0.173471,1.018178,0.956778,0.808
19,Dropout=0.3_L2=0.001,0.457132,1.114084,0.949511,0.804
3,L1=0.0001,0.630686,1.287431,0.946129,0.802


In [5]:
import numpy as np
from keras import models, layers, regularizers
from keras.datasets import reuters
from keras.utils import to_categorical
import matplotlib.pyplot as plt
import pandas as pd

# بارگذاری دیتاست
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

# تبدیل داده‌ها به بردار
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)

# داده‌های اعتبارسنجی
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = y_train[:1000]
partial_y_train = y_train[1000:]

# پارامترهای آزمایشی
dropout_rates = [0.3, 0.5, 0.7]
l1_values = [0.0001, 0.001, 0.01]
l2_values = [0.0001, 0.001, 0.01]

results = {}

# دیکشنری برای ذخیره مقادیر
metrics = {
    'Model': [],
    'Train Loss': [],
    'Val Loss': [],
    'Train Accuracy': [],
    'Val Accuracy': []
}

# تابع آموزش و ارزیابی مدل با ذخیره مقادیر
def train_model(name, model):
    print(f"Training: {name}")
    history = model.fit(partial_x_train, partial_y_train,
                        epochs=30, batch_size=512,
                        validation_data=(x_val, y_val),
                        verbose=0)
    
    # ذخیره نتایج دقت و خطا
    val_acc = history.history['val_accuracy'][-1]
    results[name] = val_acc
    
    metrics['Model'].append(name)
    metrics['Train Loss'].append(history.history['loss'][-1])
    metrics['Val Loss'].append(history.history['val_loss'][-1])
    metrics['Train Accuracy'].append(history.history['accuracy'][-1])
    metrics['Val Accuracy'].append(history.history['val_accuracy'][-1])

    return val_acc

# 🟦 فقط L2 Regularization
for val in l2_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L2={val}", model)

# 🟩 فقط L1 Regularization
for val in l1_values:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(val)))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"L1={val}", model)

# 🟨 ترکیب L1 و L2
for l1_val in l1_values:
    for l2_val in l2_values:
        reg = regularizers.l1_l2(l1=l1_val, l2=l2_val)
        model = models.Sequential()
        model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=reg))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=reg))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"L1={l1_val}_L2={l2_val}", model)

# 🟧 فقط Dropout
for rate in dropout_rates:
    model = models.Sequential()
    model.add(layers.Dense(128, activation='relu', input_shape=(10000,)))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(rate))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    train_model(f"Dropout={rate}", model)

# 🟥 Dropout + L2 Regularization
for rate in dropout_rates:
    for l2_val in l2_values:
        model = models.Sequential()
        model.add(layers.Dense(128, activation='relu', input_shape=(10000,), kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_val)))
        model.add(layers.Dropout(rate))
        model.add(layers.Dense(46, activation='softmax'))
        model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
        train_model(f"Dropout={rate}_L2={l2_val}", model)

# ذخیره کردن نتایج در یک DataFrame برای نمایش
df = pd.DataFrame(metrics)

# نمایش نتایج در یک جدول (با مرتب کردن بر اساس دقت اعتبارسنجی)
# df.sort_values(by='Val Accuracy', ascending=False, inplace=True)

# نمایش جدول
df


Training: L2=0.0001
Training: L2=0.001
Training: L2=0.01
Training: L1=0.0001
Training: L1=0.001
Training: L1=0.01
Training: L1=0.0001_L2=0.0001
Training: L1=0.0001_L2=0.001
Training: L1=0.0001_L2=0.01
Training: L1=0.001_L2=0.0001
Training: L1=0.001_L2=0.001
Training: L1=0.001_L2=0.01
Training: L1=0.01_L2=0.0001
Training: L1=0.01_L2=0.001
Training: L1=0.01_L2=0.01
Training: Dropout=0.3
Training: Dropout=0.5
Training: Dropout=0.7
Training: Dropout=0.3_L2=0.0001
Training: Dropout=0.3_L2=0.001
Training: Dropout=0.3_L2=0.01
Training: Dropout=0.5_L2=0.0001
Training: Dropout=0.5_L2=0.001
Training: Dropout=0.5_L2=0.01
Training: Dropout=0.7_L2=0.0001
Training: Dropout=0.7_L2=0.001
Training: Dropout=0.7_L2=0.01


,Model,Train Loss,Val Loss,Train Accuracy,Val Accuracy
0,L2=0.0001,0.171340,1.039125,0.958406,0.801
1,L2=0.001,0.396448,1.132429,0.954899,0.801
2,L2=0.01,1.128368,1.373345,0.831496,0.776
3,L1=0.0001,0.623848,1.259043,0.946003,0.812
4,L1=0.001,1.978270,2.049554,0.778752,0.768
5,L1=0.01,7.992329,7.902697,0.579429,0.578
6,L1=0.0001_L2=0.0001,0.653312,1.299090,0.938236,0.815
7,L1=0.0001_L2=0.001,0.813633,1.368408,0.931721,0.809
8,L1=0.0001_L2=0.01,1.316150,1.453035,0.808945,0.776
9,L1=0.001_L2=0.0001,1.980057,2.046266,0.777625,0.764
